In [1]:
# ============================================
# TAHAP 2: PREPROCESSING DENGAN DATA BESAR
# ============================================

# ========== LANGKAH 1: INSTALL LIBRARY ==========
!pip install pandas sastrawi -q

# Download stopword Bahasa Indonesia
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# ========== LANGKAH 2: IMPORT LIBRARY ==========
import pandas as pd
import re
import string
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import warnings
warnings.filterwarnings('ignore')

# ========== LANGKAH 3: LOAD DATA HASIL SCRAPING ==========
# Load file hasil scraping multi-video
print("📂 Memuat data dari youtube_comments_large.csv...")
df = pd.read_csv('youtube_comments_large.csv')

print(f"✅ Jumlah data awal: {len(df):,} komentar")
print(f"\n📊 5 data pertama:")
print(df.head())

# Cek apakah ada kolom 'comment' (pastikan nama kolomnya sesuai)
if 'comment' not in df.columns:
    print("\n⚠️ Kolom 'comment' tidak ditemukan. Cek nama kolom:")
    print(df.columns.tolist())
    # Jika kolom bernama lain, rename di sini
    # df.rename(columns={'nama_kolom_asli': 'comment'}, inplace=True)

# ========== LANGKAH 4: HAPUS DUPLIKAT ==========
print("\n🔄 Menghapus komentar duplikat...")
before = len(df)
df = df.drop_duplicates(subset=['comment'])
after = len(df)
print(f"   Dihapus {before - after:,} komentar duplikat")
print(f"   Tersisa {after:,} komentar")

# ========== LANGKAH 5: PREPROCESSING - CLEANING ==========
print("\n🧹 Membersihkan teks komentar...")

def clean_text(text):
    """
    Membersihkan teks komentar secara menyeluruh
    """
    if not isinstance(text, str):
        return ""

    # 1. Ubah ke huruf kecil
    text = text.lower()

    # 2. Hapus mention (@username)
    text = re.sub(r'@\w+', '', text)

    # 3. Hapus link (http, https, www, bit.ly, dll)
    text = re.sub(r'http\S+|www\S+|https\S+|bit\.ly\S+', '', text)

    # 4. Hapus emoji dan karakter aneh (hanya huruf, angka, spasi)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 5. Hapus angka (opsional, bisa disesuaikan)
    text = re.sub(r'\d+', '', text)

    # 6. Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Terapkan cleaning
df['comment_clean'] = df['comment'].apply(clean_text)

# Hapus komentar yang menjadi kosong setelah cleaning
before = len(df)
df = df[df['comment_clean'] != '']
after = len(df)
print(f"   Dihapus {before - after:,} komentar kosong")
print(f"   Tersisa {after:,} komentar")

# ========== LANGKAH 6: STOPWORD REMOVAL ==========
print("\n🚫 Menghapus stopword (kata tidak penting)...")

# Daftar stopword Bahasa Indonesia
indonesian_stopwords = set(stopwords.words('indonesian'))

# Tambahkan stopword kustom (kata slang/khas Indonesia)
custom_stopwords = {
    'yg', 'dgn', 'jd', 'klo', 'sdh', 'udah', 'gak', 'ga', 'nggak',
    'nya', 'si', 'nih', 'dong', 'sih', 'deh', 'yah', 'ah', 'kok',
    'kan', 'lah', 'pun', 'dan', 'juga', 'saya', 'kamu', 'dia', 'mereka',
    'kau', 'aku', 'gua', 'gue', 'lo', 'lu', 'anda', 'kalian'
}
indonesian_stopwords.update(custom_stopwords)

def remove_stopwords(text):
    """
    Menghapus stopword dari teks
    """
    words = text.split()
    words = [w for w in words if w not in indonesian_stopwords and len(w) > 1]
    return ' '.join(words)

df['comment_no_stopword'] = df['comment_clean'].apply(remove_stopwords)

# ========== LANGKAH 7: STEMMING ==========
print("\n🌳 Melakukan stemming (menghilangkan imbuhan)...")

# Buat stemmer Sastrawi
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_text(text):
    """
    Mengubah kata berimbuhan menjadi kata dasar
    """
    if len(text) < 1000:  # Untuk efisiensi, batasi panjang teks
        return stemmer.stem(text)
    return text  # skip jika terlalu panjang

# Terapkan stemming (ini proses paling lambat)
df['comment_stemmed'] = df['comment_no_stopword'].apply(stem_text)

print("   Stemming selesai!")

# ========== LANGKAH 8: PELABELAN SENTIMEN ==========
print("\n🏷️ Memberi label sentimen...")

# Perluas kamus kata positif (sesuaikan dengan domain printer/gadget)
positive_lexicon = {
    # Kata dasar positif
    'bagus', 'baik', 'hebat', 'keren', 'mantap', 'suka', 'senang', 'puas',
    'terbaik', 'recommended', 'wow', 'gokil', 'keren', 'best', 'top',
    'berkualitas', 'murah', 'cepat', 'lengkap', 'akurat', 'bermanfaat',
    'rekomendasi', 'rekomended', 'worthit', 'juara', 'istimewa', 'sempurna',
    'canggih', 'handal', 'awet', 'tahan lama', 'ekonomis', 'irit',
    # Slang positif
    'gaskeun', 'mantul', 'cakep', 'kece', 'jos', 'gaspol', 'the best',
    # Untuk produk printer
    'hasilnya bagus', 'print bagus', 'warna bagus', 'tajam', 'jelas'
}

# Perluas kamus kata negatif
negative_lexicon = {
    # Kata dasar negatif
    'jelek', 'buruk', 'parah', 'kecewa', 'benci', 'kesal', 'marah', 'sedih',
    'gagal', 'error', 'lambat', 'mahal', 'rusak', 'curang', 'bohong',
    'sampah', 'tolol', 'bodoh', 'goblok', 'payah', 'lemot', 'mengecewakan',
    'ampas', 'norak', 'mager', 'capek', 'repot', 'ngeri', 'stress',
    # Untuk produk printer
    'macet', 'tidak bisa', 'gmna sih', 'kecewa berat', 'boros', 'tinta bocor',
    'error mulu', 'lemot banget', 'rusak parah', 'gampang rusak',
    'tidak recommend', 'bilangnya bagus', 'mengecewakan sekali'
}

def label_sentiment(text):
    """
    Memberi label sentimen berdasarkan kamus positif/negatif
    Lebih canggih: hitung proporsi kata
    """
    words = text.split()
    if not words:
        return 'netral'

    # Hitung kata positif dan negatif
    pos_count = sum(1 for w in words if w in positive_lexicon)
    neg_count = sum(1 for w in words if w in negative_lexicon)

    # Jika ada kata yang sangat kuat, bisa override (contoh)
    if 'sangat' in words:
        if 'jelek' in words or 'buruk' in words:
            return 'negatif'
        if 'bagus' in words or 'keren' in words:
            return 'positif'

    # Penentuan sentimen
    if pos_count > neg_count:
        return 'positif'
    elif neg_count > pos_count:
        return 'negatif'
    else:
        return 'netral'

df['sentiment'] = df['comment_stemmed'].apply(label_sentiment)

# ========== LANGKAH 9: VERIFIKASI HASIL ==========
print("\n" + "="*60)
print("📊 DISTRIBUSI SENTIMEN")
print("="*60)

sentiment_counts = df['sentiment'].value_counts()
print(sentiment_counts)

print("\n📊 PERSENTASE:")
percentages = df['sentiment'].value_counts(normalize=True) * 100
for sentiment, pct in percentages.items():
    bar = "█" * int(pct // 2)
    print(f"   {sentiment:7s} : {pct:5.1f}% {bar}")

# ========== LANGKAH 10: TAMPILKAN CONTOH ==========
print("\n" + "="*60)
print("📝 CONTOH KOMENTAR PER SENTIMEN")
print("="*60)

print("\n🔵 POSITIF:")
for i, text in enumerate(df[df['sentiment'] == 'positif']['comment_stemmed'].head(3).values):
    print(f"   {i+1}. {text[:100]}...")

print("\n⚪ NETRAL:")
for i, text in enumerate(df[df['sentiment'] == 'netral']['comment_stemmed'].head(3).values):
    print(f"   {i+1}. {text[:100]}...")

print("\n🔴 NEGATIF:")
for i, text in enumerate(df[df['sentiment'] == 'negatif']['comment_stemmed'].head(3).values):
    print(f"   {i+1}. {text[:100]}...")

# ========== LANGKAH 11: SIMPAN DATA ==========
print("\n" + "="*60)
print("💾 MENYIMPAN DATA HASIL PREPROCESSING")
print("="*60)

# Pilih kolom yang diperlukan untuk training
df_final = df[['comment_clean', 'comment_no_stopword', 'comment_stemmed', 'sentiment']].copy()

# Simpan ke CSV
df_final.to_csv('sentiment_data_clean_large.csv', index=False)
print(f"✅ Disimpan ke 'sentiment_data_clean_large.csv'")
print(f"   Total data: {len(df_final):,} komentar")

# ========== LANGKAH 12: STATISTIK LENGKAP ==========
print("\n" + "="*60)
print("📈 STATISTIK DATA FINAL")
print("="*60)

print(f"📌 Total data siap training: {len(df_final):,}")
print(f"📌 Panjang komentar rata-rata: {df_final['comment_stemmed'].str.len().mean():.0f} karakter")
print(f"📌 Panjang komentar terpendek: {df_final['comment_stemmed'].str.len().min():,} karakter")
print(f"📌 Panjang komentar terpanjang: {df_final['comment_stemmed'].str.len().max():,} karakter")

# Cek keseimbangan data
min_class = sentiment_counts.min()
max_class = sentiment_counts.max()
imbalance_ratio = max_class / min_class if min_class > 0 else float('inf')

print(f"\n📌 Rasio ketidakseimbangan: {imbalance_ratio:.1f}:1")
if imbalance_ratio > 3:
    print("   ⚠️ Data tidak seimbang. Pertimbangkan oversampling nanti.")
else:
    print("   ✅ Data cukup seimbang.")

print("\n" + "="*60)
print("🎉 PREPROCESSING SELESAI! SIAP KE TAHAP TRAINING!")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.9 MB/s eta 0:00:00
📂 Memuat data dari youtube_comments_large.csv...
✅ Jumlah data awal: 9,702 komentar

📊 5 data pertama:
                                             comment     video_id
0                        kok seri Epson gk masuk ya?  H7H0jbdJFNo
1                            Printer sahabat pena 😂😂  H7H0jbdJFNo
2  Kak minta rekomendasi nya... Printer All in on...  jfGKlightJU
3  menurutkn brother gak ada obat,  rata rata pri...  jfGKlightJU
4  Min mau tanya bagusan mana epson L3211 dengan ...  jfGKlightJU

🔄 Menghapus komentar duplikat...
   Dihapus 195 komentar duplikat
   Tersisa 9,507 komentar

🧹 Membersihkan teks komentar...
   Dihapus 205 komentar kosong
   Tersisa 9,302 komentar

🚫 Menghapus stopword (kata tidak penting)...

🌳 Melakukan stemming (menghilangkan imbuhan)...
   Stemming selesai!

🏷️ Memberi label sentimen...

📊 DISTRIBUSI SENTIMEN
sentiment
netral     7984
positif    1037
negatif     281
Name: count